# Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory data analysis on the Brazilian E-commerce dataset.

## Requirements

To run this notebook successfully, install the required packages:

```bash
pip install numpy pandas matplotlib seaborn
```

This notebook imports those packages directly, so make sure they are available in your current Python environment before running the cells.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
from pathlib import Path

# Resolve the project root so data loading works from any working directory
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
data_path = project_root / 'data' / 'processed' / 'master_dataset.csv'

# Load the master dataset
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 1. Dataset Overview

In [ ]:
# Basic info
print("DATASET INFO:")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Data Types:")
print(df.dtypes.value_counts())

print("\nMISSING VALUES:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Sample data
print("SAMPLE DATA:")
display(df.head())

print("\nCOLUMN NAMES:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

## 2. Univariate Analysis

In [ ]:
# Numerical columns analysis
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Statistical summary for numerical columns
print("NUMERICAL COLUMNS SUMMARY:")
display(df[numerical_cols].describe())

In [ ]:
# Distribution plots for key numerical variables
key_numerical = ['price', 'freight_value', 'item_total_value', 'delivery_time_days', 
                'review_score', 'payment_value', 'product_weight_g']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for i, col in enumerate(key_numerical):
    if col in df.columns:
        df[col].hist(bins=50, ax=axes[i], alpha=0.7)
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')

# Remove empty subplots
for j in range(len(key_numerical), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for outlier detection
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i, col in enumerate(key_numerical):
    if col in df.columns and i < 8:
        df.boxplot(column=col, ax=axes[i])
        axes[i].set_title(f'Box Plot: {col}')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical variables analysis
key_categorical = ['order_status', 'customer_state', 'product_category_name', 
                  'payment_type', 'sentiment', 'order_stage']

for col in key_categorical:
    if col in df.columns:
        print(f"\n{col.upper()} VALUE COUNTS:")
        value_counts = df[col].value_counts()
        print(value_counts.head(10))
        
        # Plot
        plt.figure(figsize=(10, 6))
        if len(value_counts) > 15:
            value_counts.head(15).plot(kind='bar')
            plt.title(f'Top 15 {col}')
        else:
            value_counts.plot(kind='bar')
            plt.title(f'{col} Distribution')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 3. Bivariate Analysis

In [ ]:
# Correlation matrix for numerical variables
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f')
plt.title('Correlation Matrix of Numerical Variables')
plt.tight_layout()
plt.show()

In [ ]:
# Price vs Review Score
if 'price' in df.columns and 'review_score' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='review_score', y='price')
    plt.title('Price Distribution by Review Score')
    plt.yscale('log')
    plt.show()
    
    # Statistical summary
    print("PRICE BY REVIEW SCORE:")
    print(df.groupby('review_score')['price'].agg(['mean', 'median', 'std']).round(2))

In [ ]:
# Delivery time vs Review Score
if 'delivery_time_days' in df.columns and 'review_score' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='review_score', y='delivery_time_days')
    plt.title('Delivery Time Distribution by Review Score')
    plt.show()
    
    print("DELIVERY TIME BY REVIEW SCORE:")
    print(df.groupby('review_score')['delivery_time_days'].agg(['mean', 'median', 'std']).round(2))

In [ ]:
# Payment type vs Order value
if 'payment_type' in df.columns and 'payment_value' in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='payment_type', y='payment_value')
    plt.title('Payment Value Distribution by Payment Type')
    plt.xticks(rotation=45)
    plt.yscale('log')
    plt.show()
    
    print("PAYMENT VALUE BY PAYMENT TYPE:")
    print(df.groupby('payment_type')['payment_value'].agg(['mean', 'median', 'count']).round(2))

## 4. Geographic Analysis

In [ ]:
# State-wise analysis
if 'customer_state' in df.columns:
    state_analysis = df.groupby('customer_state').agg({
        'order_id': 'count',
        'payment_value': ['mean', 'sum'],
        'review_score': 'mean',
        'delivery_time_days': 'mean'
    }).round(2)
    
    state_analysis.columns = ['Order_Count', 'Avg_Payment', 'Total_Revenue', 
                             'Avg_Review_Score', 'Avg_Delivery_Days']
    state_analysis = state_analysis.sort_values('Order_Count', ascending=False)
    
    print("TOP 10 STATES BY ORDER COUNT:")
    display(state_analysis.head(10))
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Orders by state
    state_analysis.head(10)['Order_Count'].plot(kind='bar', ax=axes[0,0])
    axes[0,0].set_title('Top 10 States by Order Count')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # Revenue by state
    state_analysis.head(10)['Total_Revenue'].plot(kind='bar', ax=axes[0,1])
    axes[0,1].set_title('Top 10 States by Total Revenue')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # Average review score by state
    state_analysis.head(10)['Avg_Review_Score'].plot(kind='bar', ax=axes[1,0])
    axes[1,0].set_title('Top 10 States by Average Review Score')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # Average delivery time by state
    state_analysis.head(10)['Avg_Delivery_Days'].plot(kind='bar', ax=axes[1,1])
    axes[1,1].set_title('Top 10 States by Average Delivery Time')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 5. Product Analysis

In [ ]:
# Product category analysis
if 'product_category_name_english' in df.columns:
    category_analysis = df.groupby('product_category_name_english').agg({
        'order_id': 'count',
        'price': 'mean',
        'payment_value': 'sum',
        'review_score': 'mean',
        'freight_value': 'mean'
    }).round(2)
    
    category_analysis.columns = ['Order_Count', 'Avg_Price', 'Total_Revenue', 
                                'Avg_Review_Score', 'Avg_Freight']
    category_analysis = category_analysis.sort_values('Order_Count', ascending=False)
    
    print("TOP 15 PRODUCT CATEGORIES:")
    display(category_analysis.head(15))
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Top categories by order count
    category_analysis.head(15)['Order_Count'].plot(kind='barh', ax=axes[0,0])
    axes[0,0].set_title('Top 15 Categories by Order Count')
    
    # Top categories by revenue
    category_analysis.head(15)['Total_Revenue'].plot(kind='barh', ax=axes[0,1])
    axes[0,1].set_title('Top 15 Categories by Total Revenue')
    
    # Average price by category
    category_analysis.head(15)['Avg_Price'].plot(kind='barh', ax=axes[1,0])
    axes[1,0].set_title('Top 15 Categories by Average Price')
    
    # Average review score by category
    category_analysis.head(15)['Avg_Review_Score'].plot(kind='barh', ax=axes[1,1])
    axes[1,1].set_title('Top 15 Categories by Average Review Score')
    
    plt.tight_layout()
    plt.show()

## 6. Time Series Analysis

In [ ]:
# Convert date columns
date_columns = ['order_purchase_timestamp', 'order_delivered_customer_date', 
               'review_creation_date']

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Extract time features
if 'order_purchase_timestamp' in df.columns:
    df['order_year'] = df['order_purchase_timestamp'].dt.year
    df['order_month'] = df['order_purchase_timestamp'].dt.month
    df['order_day_of_week'] = df['order_purchase_timestamp'].dt.dayofweek
    df['order_hour'] = df['order_purchase_timestamp'].dt.hour

In [ ]:
# Monthly trends
if 'order_purchase_timestamp' in df.columns:
    monthly_trends = df.groupby(df['order_purchase_timestamp'].dt.to_period('M')).agg({
        'order_id': 'count',
        'payment_value': ['sum', 'mean'],
        'review_score': 'mean'
    })
    
    monthly_trends.columns = ['Order_Count', 'Total_Revenue', 'Avg_Order_Value', 'Avg_Review_Score']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Monthly order count
    monthly_trends['Order_Count'].plot(ax=axes[0,0])
    axes[0,0].set_title('Monthly Order Count Trend')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # Monthly revenue
    monthly_trends['Total_Revenue'].plot(ax=axes[0,1])
    axes[0,1].set_title('Monthly Revenue Trend')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # Average order value
    monthly_trends['Avg_Order_Value'].plot(ax=axes[1,0])
    axes[1,0].set_title('Monthly Average Order Value Trend')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # Average review score
    monthly_trends['Avg_Review_Score'].plot(ax=axes[1,1])
    axes[1,1].set_title('Monthly Average Review Score Trend')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Day of week and hour analysis
if 'order_day_of_week' in df.columns and 'order_hour' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Orders by day of week
    day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    dow_counts = df['order_day_of_week'].value_counts().sort_index()
    dow_counts.index = day_names
    dow_counts.plot(kind='bar', ax=axes[0])
    axes[0].set_title('Orders by Day of Week')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Orders by hour
    df['order_hour'].value_counts().sort_index().plot(kind='bar', ax=axes[1])
    axes[1].set_title('Orders by Hour of Day')
    axes[1].tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.show()

## 7. Customer Behavior Analysis

In [ ]:
# Customer analysis
if 'customer_unique_id' in df.columns:
    customer_analysis = df.groupby('customer_unique_id').agg({
        'order_id': 'count',
        'payment_value': ['sum', 'mean'],
        'review_score': 'mean',
        'delivery_time_days': 'mean'
    }).round(2)
    
    customer_analysis.columns = ['Order_Count', 'Total_Spent', 'Avg_Order_Value', 
                                'Avg_Review_Score', 'Avg_Delivery_Days']
    
    print("CUSTOMER BEHAVIOR SUMMARY:")
    print(f"Total Unique Customers: {len(customer_analysis)}")
    print(f"Repeat Customers: {(customer_analysis['Order_Count'] > 1).sum()}")
    print(f"Repeat Customer Rate: {(customer_analysis['Order_Count'] > 1).mean():.2%}")
    
    # Customer segmentation by order count
    plt.figure(figsize=(10, 6))
    customer_analysis['Order_Count'].value_counts().sort_index().plot(kind='bar')
    plt.title('Customer Distribution by Number of Orders')
    plt.xlabel('Number of Orders')
    plt.ylabel('Number of Customers')
    plt.show()
    
    # Top customers
    print("\nTOP 10 CUSTOMERS BY TOTAL SPENT:")
    display(customer_analysis.sort_values('Total_Spent', ascending=False).head(10))

## 8. Key Insights and Summary

In [ ]:
# Summary statistics
print("=" * 60)
print("KEY BUSINESS INSIGHTS")
print("=" * 60)

# Overall metrics
total_orders = df['order_id'].nunique()
total_customers = df['customer_unique_id'].nunique() if 'customer_unique_id' in df.columns else 'N/A'
total_revenue = df['payment_value'].sum() if 'payment_value' in df.columns else 'N/A'
avg_order_value = df['payment_value'].mean() if 'payment_value' in df.columns else 'N/A'
avg_review_score = df['review_score'].mean() if 'review_score' in df.columns else 'N/A'

print(f"BUSINESS OVERVIEW:")
print(f"   • Total Orders: {total_orders:,}")
print(f"   • Total Customers: {total_customers:,}" if total_customers != 'N/A' else f"   • Total Customers: {total_customers}")
print(f"   • Total Revenue: R$ {total_revenue:,.2f}" if total_revenue != 'N/A' else f"   • Total Revenue: {total_revenue}")
print(f"   • Average Order Value: R$ {avg_order_value:.2f}" if avg_order_value != 'N/A' else f"   • Average Order Value: {avg_order_value}")
print(f"   • Average Review Score: {avg_review_score:.2f}/5" if avg_review_score != 'N/A' else f"   • Average Review Score: {avg_review_score}")

# Top performing categories
if 'product_category_name_english' in df.columns:
    top_category = df['product_category_name_english'].value_counts().index[0]
    print(f"\nTOP PERFORMING CATEGORY: {top_category}")

# Geographic insights
if 'customer_state' in df.columns:
    top_state = df['customer_state'].value_counts().index[0]
    print(f"\nTOP STATE BY ORDERS: {top_state}")

# Payment insights
if 'payment_type' in df.columns:
    top_payment = df['payment_type'].value_counts().index[0]
    print(f"\nMOST POPULAR PAYMENT METHOD: {top_payment}")

# Delivery insights
if 'delivery_time_days' in df.columns:
    avg_delivery = df['delivery_time_days'].mean()
    print(f"\nAVERAGE DELIVERY TIME: {avg_delivery:.1f} days")

print("\n" + "=" * 60)
print("EDA COMPLETED SUCCESSFULLY!")
print("=" * 60)